<a href="https://colab.research.google.com/github/CECS-AI/ai101-assignment5-prangara-vols/blob/main/Assignment5a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from google.colab import userdata, drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


### **Setup and Data Loading**

This section handles the initial setup, including mounting Google Drive, installing necessary Python packages, configuring the Gemini API, and loading the dataset into a string variable.

In [14]:
from google.colab import userdata, drive
drive.mount("/content/drive", force_remount=True)

!pip install -q -U crewai
!pip install -q -U duckduckgo-search
!pip install -q -U crewai[tools]
!pip install -q -U crewai[hf] # Install Hugging Face integration for CrewAI
!pip install -q -U litellm # Install LiteLLM fallback for HuggingFace models

import os
from crewai import Agent, Task, Crew, Process, LLM
# import google.generativeai as genai # No longer needed for Hugging Face

from crewai.tools import BaseTool

# Retrieve Hugging Face API key from Colab secrets
# IMPORTANT: Add your Hugging Face token to Colab's Secrets manager, named 'HF_TOKEN'
HF_TOKEN = userdata.get('HF_TOKEN')

# Configure the API key - not needed for genai directly with CrewAI[hf]
# The following lines are commented out/removed as Gemini is no longer being used.
# genai.configure(api_key=gemini_api_key)

# IMPORTANT: Update with the correct path to your Assignment 1 dataset (e.g., 'my_assignment1_data.csv')
file_path = '/content/drive/My Drive/Colab Notebooks/Hitters.csv'

try:
    with open(file_path, 'r') as f:
        csv_string = f.read()
    print("File content read successfully!")
    print(csv_string[:500]) # Print the first 500 characters to show the content
except FileNotFoundError:
    print(f"File not found at {file_path}. Please check the path and try again.")
except Exception as e:
    print(f"An error occurred: {e}")

Mounted at /content/drive
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.14 requires pydantic==2.12.5, but you have pydantic 2.11.10 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.34.1 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.11.10 which is incompatible.
bigframes 2.39.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

### **Hugging Face LLM Configuration**

This cell configures the Large Language Model (LLM) that the agents will use, leveraging Hugging Face models.

- `from crewai.utilities import get_llm`: Imports the utility function to easily configure different LLMs.
- `hf_llm = get_llm(...)`: Initializes the LLM object for CrewAI.
- `llm='hf'`: Specifies that we are using a Hugging Face model.
- `model='google/flan-t5-large'`: Specifies the particular Hugging Face model to use. You can change this to other available models on Hugging Face (e.g., 'tiiuae/falcon-7b-instruct', 'HuggingFaceH4/zephyr-7b-beta'). Please ensure the model is suitable for your task and accessible for inference.
- `hf_api_key=HF_TOKEN`: Passes your configured Hugging Face API key for authentication. This token is retrieved from Colab's secrets.
- `temperature=0.0`: Sets the creativity level of the model. A temperature of 0.0 makes the output more deterministic and consistent, which is often preferred for analytical tasks.

In [18]:
import os

# CrewAI uses LiteLLM under the hood, which looks for this environment variable
os.environ["HUGGINGFACE_API_KEY"] = HF_TOKEN

# Define the Hugging Face model using the 'provider/model' string format
# Qwen2.5-7B-Instruct is widely supported on the HF free inference API for chat completions
hf_llm = 'huggingface/Qwen/Qwen2.5-7B-Instruct'

print(f"LLM configured to use: {hf_llm}")
print("You can now run the next cells to create your agents!")

LLM configured to use: huggingface/Qwen/Qwen2.5-7B-Instruct
You can now run the next cells to create your agents!


### **Custom Tool Definition and Data Analyst Agent Setup**

This section defines a custom tool that allows agents to directly access the loaded CSV data, and then sets up the `data_analyst` agent and its primary task.

#### **`CsvStringReaderTool`**

- `class CsvStringReaderTool(BaseTool):`: Defines a custom tool class inheriting from `crewai.tools.BaseTool`.
- `name`, `description`: Provide metadata about the tool for the agent to understand its purpose.
- `csv_data: str`: A field to hold the CSV content, passed during instantiation.
- `_run(self, query: str = None) -> str`: The method executed when the tool is used. It simply returns the `csv_data`.

#### **`data_analyst` Agent**

- `data_analyst = Agent(...)`: Instantiates an agent with specific characteristics:
  - `role`: The primary role of the agent.
  - `goal`: What the agent aims to achieve.
  - `backstory`: Provides context and personality to the agent.
  - `verbose=True`: Enables detailed logging of the agent's thought process.
  - `allow_delegation=False`: Prevents this agent from delegating parts of its task to other agents.
  - `llm=gemini_llm`: Assigns the configured Gemini LLM to this agent.
  - `tools=[csv_reader_tool]`: Provides the `CsvStringReaderTool` to the agent, allowing it to read the CSV data.

In [20]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
import pandas as pd
import io

# Define a custom tool to read a summarized version of the CSV
class CsvStringReaderTool(BaseTool):
    name: str = "CSV String Reader"
    description: str = "Reads the provided CSV data summary. Useful for agents needing insights without exceeding token limits."
    csv_data: str

    def _run(self, query: str = None) -> str:
        """Reads the CSV data summary."""
        return self.csv_data

# The full CSV is too large, so we create a condensed summary
df = pd.read_csv(io.StringIO(csv_string))
data_summary = f"Data Sample (first 5 rows):\n{df.head(5).to_csv(index=False)}\n\nSummary Statistics:\n{df.describe().to_csv()}\n\nMissing Values:\n{df.isnull().sum().to_string()}"

# Instantiate the custom tool with the condensed summary
csv_reader_tool = CsvStringReaderTool(csv_data=data_summary)

# Define one of your agents
data_analyst = Agent(
    role='Baseball Data Analyst',
    goal='Analyze the provided baseball dataset to extract key insights, performance metrics, and patterns.',
    backstory='A meticulous data analyst with a keen eye for detail, specializing in uncovering hidden trends in sports statistics and generating clear summaries from raw data.',
    verbose=True,
    allow_delegation=False,
    llm=hf_llm, # Use the configured Hugging Face LLM
    tools=[csv_reader_tool] # Add the custom tool to the agent
)

# Define the task for the agent
analyze_data_task = Task(
    description=f"""
  Using the provided CSV data summary (Hitters.csv), perform a comprehensive analysis.
  Identify the main entities, relevant performance metrics (like Hits, Home Runs, RBI), and any significant patterns or anomalies.
  Summarize your findings in a structured report, highlighting key observations.
  Focus on the relationship between player performance statistics and their salaries.
  """,
    expected_output='A detailed report summarizing the key findings from the baseball dataset analysis, including trends, patterns, and important metrics, presented in a clear and organized manner.',
    agent=data_analyst
)

### **Stock Predictor Agent and Crew Setup**

This section defines a `stock_predictor` agent, a task for it, and then orchestrates both agents and their tasks into a `Crew` to perform the overall assignment.

#### **`stock_predictor` Agent**

- `stock_predictor = Agent(...)`: Instantiates another agent, similar to the `data_analyst`.
  - Its `role`, `goal`, and `backstory` should be tailored to its specific function of predicting stock prices or analyzing trends based on the data.

#### **`predict_stock_price_task`**

- `predict_stock_price_task = Task(...)`: Defines the task for the `stock_predictor` agent.
  - `description`: Details what the agent needs to do using the available data.
  - `expected_output`: Describes the format and content of the task's final output.
  - `agent=stock_predictor`: Assigns this task to the `stock_predictor` agent.
  - `context=[analyze_data_task]`: Crucially, this provides the output of the `analyze_data_task` (from the `data_analyst`) as context for the `stock_predictor`, enabling a multi-step workflow.

#### **`financial_analysis_and_prediction_crew`**

- `financial_analysis_and_prediction_crew = Crew(...)`: Creates the `Crew` object that manages the agents and tasks.
  - `agents=[data_analyst, stock_predictor]`: Lists all agents participating in this crew.
  - `tasks=[analyze_data_task, predict_stock_price_task]`: Lists all tasks to be executed by the crew.
  - `process=Process.hierarchical`: Specifies that tasks are executed in a hierarchical manner, often meaning one agent's output feeds another's input.
  - `manager_llm=gemini_llm`: Assigns an LLM to manage the overall workflow and agent interactions within the crew.

#### **Execution**

- `print("## Starting the ... Crew")`: An informative print statement.
- `result = financial_analysis_and_prediction_crew.kickoff()`: Initiates the entire workflow, causing the agents to perform their tasks sequentially.
- `print("## Final Report:")` and `print(result)`: Display the final consolidated output from the crew's work.

In [22]:
# Define another agent
sports_predictor = Agent(
    role='Sports Insight Synthesizer',
    goal='Synthesize analytical findings into actionable recommendations and salary predictions.',
    backstory='An expert in synthesizing complex baseball statistics into clear, strategic recommendations. This agent excels at connecting dots and providing forward-looking insights on player value.',
    verbose=True,
    allow_delegation=False,
    llm=hf_llm # Use the configured Hugging Face LLM
)

# Define the task for the insight generation agent
predict_salary_task = Task(
    description=f"""
  Based on the detailed analysis provided by the Data Analyst, synthesize the key insights.
  Formulate predictions regarding player value and expected salaries based on their historical performance metrics.
  Present these in a concise, action-oriented summary.
  """,
    expected_output="A summary of synthesized insights and actionable predictions regarding player salaries based on the data analyst's report, clearly outlining future implications for team management and drafting strategies.",
    agent=sports_predictor,
    context=[analyze_data_task] # Add the analysis task as context
)


# Create a crew with both agents and tasks
assignment5a_crew = Crew(
    agents=[data_analyst, sports_predictor],
    tasks=[analyze_data_task, predict_salary_task],
    process=Process.sequential # Changed to sequential to avoid manager LLM errors with free tier APIs
)

# Kick off the crew's work
print("## Starting the Assignment 5a Crew")
result = assignment5a_crew.kickoff()

# Print the results
print("## Final Report:")
print(result)

## Starting the Assignment 5a Crew


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sports Insight Synthesizer                                                                              │
│                                                                                                                 │
│  Task:                                                                                                          │
│    Based on the detailed analysis provided by the Data Analyst, synthesize the key insights.                    │
│    Formulate predictions regarding player value and expected salaries based on their historical performance     │
│  metrics.                                                                                                       │
│    Present these in a concise, action-oriented summary.                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sports Insight Synthesizer                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Synthesized Insights and Actionable Predictions                                                            │
│                                                                                                                 │
│  #### Key Observations:                                                                                         │
│  1. **Positive Correlation Between Performance Metrics and Salaries:**                                          │
│     - Players with higher performance metrics (AtBat, Hits, HmRun, Runs, RBI, Walks) tend to have higher        │
│  salaries.                                                                                                      │
│     - This indicates that performance is a significant factor in determining player value and, consequently,    │
│  their salaries.                                                                                                │
│                                                                                                                 │
│  2. **Non-Linear Relationship:**                                                                                │
│     - The relationship between performance and salary is not perfectly linear. There are instances where        │
│  players with high performance metrics have relatively low salaries, and vice versa.                            │
│     - This suggests that other factors, such as market demand, team strategy, and player contracts, also play   │
│  a role in determining salaries.                                                                                │
│                                                                                                                 │
│  3. **Missing Salary Values:**                                                                                  │
│     - The dataset contains 59 missing salary values, which may introduce bias in the analysis and should be     │
│  addressed through imputation or other methods.                                                                 │
│                                                                                                                 │
│  #### Actionable Predictions:                                                                                   │
│  1. **Salary Projections:**                                                                                     │
│     - **High-Performance Players:** Players with high performance metrics (e.g., > 100 Hits, > 10 Home Runs, >  │
│  50 RBIs) are likely to command higher salaries. For example, a player with 120 Hits, 15 Home Runs, and 60      │
│  RBIs is expected to have a salary in the range of $3-4 million, assuming a linear relationship and no other    │
│  factors.                                                                                                       │
│     - **Mid-Range Players:** Players with moderate performance metrics (e.g., 80-100 Hits, 5-10 Home Runs,      │
│  30-50 RBIs) are expected to have salaries in the range of $1-2 million.                                        │
│     - **Low-Performance Players:** Players with low performance metrics (e.g., < 80 Hits, < 5 Home Runs, < 30   │
│  RBIs) are expected to have salaries below $1 million.                                                          │
│                                                        

## Final Report:
### Synthesized Insights and Actionable Predictions

#### Key Observations:
1. **Positive Correlation Between Performance Metrics and Salaries:**
   - Players with higher performance metrics (AtBat, Hits, HmRun, Runs, RBI, Walks) tend to have higher salaries.
   - This indicates that performance is a significant factor in determining player value and, consequently, their salaries.

2. **Non-Linear Relationship:**
   - The relationship between performance and salary is not perfectly linear. There are instances where players with high performance metrics have relatively low salaries, and vice versa.
   - This suggests that other factors, such as market demand, team strategy, and player contracts, also play a role in determining salaries.

3. **Missing Salary Values:**
   - The dataset contains 59 missing salary values, which may introduce bias in the analysis and should be addressed through imputation or other methods.

#### Actionable Predictions:
1. **Salary Projection